# Train D4-ORQB Model V from scratch

This clean launcher runs classical-context pretraining followed by a 50-epoch TorchQuantum stage. It uses Model V development data only; the official test set is never referenced. All runtime paths are intentionally blank.

In [ ]:
from pathlib import Path
import os
import shlex
import shutil
import subprocess
import sys

import torch

try:
    import torchquantum as tq
except Exception as exc:
    raise RuntimeError("Install the pinned TorchQuantum dependency before running this notebook") from exc


def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "d4_orqb" / "main.py").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the deeplense-quantum repository")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
print({"repository": str(REPO_ROOT), "torchquantum": getattr(tq, "__version__", "pinned-git")})

## Runtime configuration

Set the environment variables or replace the blank strings only on the GPU machine. DEVELOPMENT_ROOT must point to Model_V, with axion, cdm, and no_sub class directories. Point RESULTS_ROOT at a fresh writable results root. The generated best.pt stays in the run directory until it is explicitly reviewed and selected.

In [ ]:
DATASET_ID = "model_v"
QUANTUM_EPOCHS = 50

DEVELOPMENT_ROOT = os.environ.get("D4_ORQB_DEVELOPMENT_ROOT", "")
CACHE_ROOT = os.environ.get("D4_ORQB_CACHE_ROOT", "")
RUN_ROOT = os.environ.get("D4_ORQB_RUN_ROOT", "")
RESULTS_ROOT = os.environ.get("D4_ORQB_RESULTS_ROOT", "")
RUN_NAME = os.environ.get("D4_ORQB_RUN_NAME", "")

PRETRAIN_LEARNING_RATE = 4e-3
PRETRAIN_CORE_LEARNING_RATE = 6e-3
ENCODER_LEARNING_RATE = 5e-4
LEARNING_RATE = 3e-3
CORE_LEARNING_RATE = 5e-3

In [ ]:
EXPECTED_CLASSES = {"axion", "cdm", "no_sub"}


def required_path(raw, name):
    if not isinstance(raw, str) or not raw.strip():
        raise ValueError(f"Fill {name} before running")
    return Path(raw).expanduser().resolve()


def validate_class_root(root, name):
    if not root.is_dir():
        raise FileNotFoundError(f"{name} not found: {root}")
    classes = {path.name for path in root.iterdir() if path.is_dir()}
    if classes != EXPECTED_CLASSES:
        raise RuntimeError(f"{name} classes are {sorted(classes)}; expected {sorted(EXPECTED_CLASSES)}")
    missing = [class_name for class_name in sorted(EXPECTED_CLASSES) if not any((root / class_name).glob("*.npy"))]
    if missing:
        raise RuntimeError(f"{name} class directories contain no .npy files: {missing}")


if not RUN_NAME.strip() or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("Set RUN_NAME to one new directory name")

DEVELOPMENT_PATH = required_path(DEVELOPMENT_ROOT, "DEVELOPMENT_ROOT")
CACHE_PATH = required_path(CACHE_ROOT, "CACHE_ROOT")
RUN_ROOT_PATH = required_path(RUN_ROOT, "RUN_ROOT")
RESULTS_ROOT_PATH = required_path(RESULTS_ROOT, "RESULTS_ROOT")
OUTPUT_DIR = RUN_ROOT_PATH / DATASET_ID / RUN_NAME
RESULTS_DIR = RESULTS_ROOT_PATH / DATASET_ID / RUN_NAME
STAGE_DIR = OUTPUT_DIR / f"quantum_seed2_{QUANTUM_EPOCHS}ep"

if OUTPUT_DIR == RESULTS_DIR:
    raise ValueError("RUN_ROOT and RESULTS_ROOT must produce distinct run directories")

validate_class_root(DEVELOPMENT_PATH, "Model V development root")
if not hasattr(tq, "QuantumDevice"):
    raise RuntimeError("The installed torchquantum module is not the expected training backend")
if not torch.cuda.is_available():
    raise RuntimeError("D4-ORQB training requires a CUDA-capable GPU")
for path, name in ((OUTPUT_DIR, "run output"), (RESULTS_DIR, "curated results")):
    if path.exists():
        raise FileExistsError(f"Refusing to reuse {name} directory: {path}")

for root in (CACHE_PATH, RUN_ROOT_PATH, RESULTS_ROOT_PATH):
    root.mkdir(parents=True, exist_ok=True)
print({"dataset": DATASET_ID, "development": str(DEVELOPMENT_PATH), "output": str(OUTPUT_DIR), "gpu": torch.cuda.get_device_name(0)})

## Run both training stages

The command explicitly requests 50 quantum epochs and the current learning-rate schedule. It passes no test path or test-evaluation flag.

In [ ]:
COMMAND = [
    sys.executable, "-m", "d4_orqb.main",
    "--dataset-id", DATASET_ID,
    "--development-root", str(DEVELOPMENT_PATH),
    "--cache-root", str(CACHE_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--stage", "all",
    "--quantum-epochs", str(QUANTUM_EPOCHS),
    "--pretrain-learning-rate", str(PRETRAIN_LEARNING_RATE),
    "--pretrain-core-learning-rate", str(PRETRAIN_CORE_LEARNING_RATE),
    "--encoder-learning-rate", str(ENCODER_LEARNING_RATE),
    "--learning-rate", str(LEARNING_RATE),
    "--core-learning-rate", str(CORE_LEARNING_RATE),
]
print("PYTHONPATH=src", shlex.join(COMMAND))

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_ROOT) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
env["PYTHONDONTWRITEBYTECODE"] = "1"
env["PYTHONUNBUFFERED"] = "1"
subprocess.run(COMMAND, cwd=REPO_ROOT, env=env, check=True)

## Curate the selected validation artifacts

After a successful fresh run, validate and report the generated best.pt candidate for later explicit review, then copy only the validation metrics and validation ROC curve into a fresh results directory. These are development-validation results, not official-test metrics.

In [ ]:
BEST_CHECKPOINT_CANDIDATE = STAGE_DIR / "best.pt"
CURATED_SOURCES = {
    STAGE_DIR / "validation_metrics.md": RESULTS_DIR / "metrics.md",
    STAGE_DIR / "validation_roc_curve.png": RESULTS_DIR / "roc_curve.png",
}
required_artifacts = (BEST_CHECKPOINT_CANDIDATE, *CURATED_SOURCES)
missing = [str(path) for path in required_artifacts if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Training completed without required curated artifacts: {missing}")
RESULTS_DIR.mkdir(parents=True, exist_ok=False)
for source, destination in CURATED_SOURCES.items():
    shutil.copy2(source, destination)
CURATED_METRICS = RESULTS_DIR / "metrics.md"
CURATED_METRICS.write_text(
    CURATED_METRICS.read_text(encoding="utf-8").replace(
        "`validation_roc_curve.png`", "`roc_curve.png`"
    ),
    encoding="utf-8",
)
print({"checkpoint_candidate_for_review": str(BEST_CHECKPOINT_CANDIDATE), "metrics": str(RESULTS_DIR / "metrics.md"), "roc_curve": str(RESULTS_DIR / "roc_curve.png")})

The generated checkpoint remains in the ignored run directory. Review the completed run before explicitly selecting a checkpoint for weights/; the notebook does not promote it automatically.